## Intermediate Machine Learning: Assignment 1

**Deadline**

Assignment 1 is due Thursday, October 1 at 11:59 pm. Late work will not be accepted as per the course policies (see the syllabus on Canvas).

Directly sharing answers is not okay, but discussing problems with the course staff or with other students is encouraged. All problems will be given full credit if a significant attempt is made at a solution, whether or not the solution is complete or fully correct. This policy is intended to allow students the freedom to get the most learning value out of the assignment, and to lower incentives to use AI. According to course policy any use of an AI system such as ChatGPT, Claude, or Copilot must be acknowledged.

You should start early so that you have time to get help if you're stuck. The drop-in office hours schedule can be found on Canvas. You can also post questions or start discussions on Ed Discussion. The assignment may look long at first glance, but the problems are broken up into steps that should help you to make steady progress.

**Submission**

Submit your assignment as a .pdf on Gradescope. You can access Gradescope through Canvas on the left-side of the class home page. The problems in each homework assignment are numbered. Note: When submitting on Gradescope, please select the correct pages of your pdf that correspond to each problem. This will allow graders to more easily find your complete solution to each problem.

To produce the .pdf, please do the following in order to preserve the cell structure of the notebook:

Go to "File" at the top-left of your Jupyter Notebook
Under "Download as", select "HTML (.html)"
After the .html has downloaded, open it and then select "File" and "Print" (note you will not actually be printing)
From the print window, select the option to save as a pdf.  You may find it helpful to
use this [converter notebook](https://colab.research.google.com/github/YData123/sds365-fa26/blob/main/assignments/Convert_ipynb_to_HTML_in_Colab.ipynb).

**Topics**

 * Lasso
 * Bias-variance decomposition
 * Mercer kernels
 * LOOCV for kernel smoothing and ridge regression

This assignment will also help to solidify your Python and Jupyter notebook skills.


### Problem 1 Exploring LASSO / Ridge / OLS Via Simulation (20 points)

In this problem, we explore the behavior of the LASSO / Ridge / OLS estimators via an empirical Monte Carlo Simulation.  We generate synthetic datasets by [bootstrapping](https://en.wikipedia.org/wiki/Bootstrapping_(statistics)) from the [UCI abalone](https://archive.ics.uci.edu/dataset/1/abalone) dataset, which consists of various measurements of Abalone sea snails. A picture of an Abalone shell is shown below.  

<a href="https://en.wikipedia.org/wiki/Abalone"><img src="https://upload.wikimedia.org/wikipedia/commons/0/0b/AbaloneInside.jpg?utm_source=commons.wikimedia.org&utm_campaign=index&utm_content=thumbnail_unscaled" width="300" style="margin:20px 30px 20px 5px"></a>



> CC BY-SA 3.0, https://commons.wikimedia.org/w/index.php?curid=203608

 The UCI repository page for the dataset explains:

>  The age of abalone is determined by cutting the shell through the cone, staining it, and counting the number of rings through a microscope -- a boring and time-consuming task.  

We want to try to develop models to predict the age of an Abalone based on a variety of easier-to-measure features.  The other features in the dataset are as follows:

- `Sex`	- The sex of the abalone snail (male, female, or infant)
- `Length`	- length of the shell
- `Diameter`	- diameter of the shell
- `Height` - diameter of the snail shell
- `Whole_weight`	- weight of the abalone
- `Shucked_weight`	- weight of meat from the abalone
- `Viscera_weight`	- weight of intestines / viscera
- `Shell_weight`	- weight of the shell

For this task, we also add a number of irrelevant noise variables, labelled "Noise 1", "Noise 2", and so on. Theses variables are drawn from a standard normal distribution, and included to demonstrate how the different methods perform at variable selection, and which methods do not perform well in the presence of many irrelevant noise variables.

### 1.1 Plot Regularization Paths

First, we define a function that allows us to sample rows from the Abalone dataset. Bootstrapping allows us to understand the distribution of the regression parameters when sampling from rows of the Abalone dataset, which will approximate the sampling distribution of these estimators.  Try calling the function with different parameters to get a sense of its behavior.


In [ ]:
!pip install ucimlrepo
import pandas as pd
import numpy as np
from sklearn import linear_model
from sklearn.linear_model import LinearRegression, Lasso, Ridge, LassoCV, RidgeCV
from tqdm import tqdm
import matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo

# fetch dataset
abalone = fetch_ucirepo(id=1)


X_abalone = abalone.data.features.copy()
X_abalone["Sex"] = (X_abalone["Sex"] == "M").astype(int)
y_abalone = abalone.data.targets.iloc[:, 0].copy()

X_abalone = (X_abalone - X_abalone.mean()) / X_abalone.std()
y_abalone = (y_abalone - y_abalone.mean()) / y_abalone.std()

ols_regression = linear_model.LinearRegression()
ols_regression.fit(X_abalone,y_abalone)

population_coefficients = np.concatenate([
    np.zeros(20),
    ols_regression.coef_.ravel(),
])


def simulate_data(n=10_000, p=20, rho=None):
    rng = np.random.default_rng()

    # Sample matched feature–outcome rows with replacement.
    idx = rng.integers(0, len(X_abalone), size=n)
    X_real = X_abalone.iloc[idx].reset_index(drop=True)
    y = y_abalone.iloc[idx].reset_index(drop=True).rename("y")

    # Generate fresh IID N(0, 1) noise for every sampled row.
    X_noise = pd.DataFrame(
        rng.standard_normal((n, p)),
        columns=[f"Noise {j + 1}" for j in range(p)],
    )

    # Combine noise variables and variables from the Abalone Dataset
    X = pd.concat([X_noise, X_real], axis=1)

    return X, y



Then use this function to generate an example dataset with 100 observations and 20 noise variables. Using code from the [Lasso Example Notebook](https://colab.research.google.com/github/YData123/sds365-fa26/blob/main/demos/lasso/lasso-example.ipynb), plot the regularization paths for both LASSO and Ridge. You can use the Lasso class from the sklearn.linear_model package. Plot the parameter paths with the regularization level (alpha in the code) on the log-scale, as done in the lasso demo code from class. (As always, be sure to label your axes).

Answer the following questions:

- How do the LASSO vs Ridge Regularization paths differ? Do you expect either one of the two penalized methods to do better in this setting? Why?
- What is the level of regularization selected by cross-validation for both regularized regressions? Which coefficients are nonzero at these values?


In [ ]:
# Space for your code and answers here

### 1.2 Fitting Regressions

Now we want to rigorously assess the behavior of Lasso, Ridge, and OLS via simulation. Help complete the following function which runs a simulation, and reports the coefficients produced by Lasso, Ridge, and OLS in a neat dataframe.

Below, we show an example of the function output we are looking for. The columns are as follows:

- *LASSO* gives the coefficient values returned by a cross-validated Lasso (using `sklearn.linear_model.LassoCV`)
  - We suggest setting `max_iter = 10000` in order to limit convergence warnings
- *Ridge* gives the coefficient values returned by a cross-validated Ridge Regression (using `sklearn.linear_model.RidgeCV`)
- *OLS* gives the coefficient values returned by an OLS Regression (using `sklearn.linear_model.LinearRegression`)

We also return the paramters used for the simulation in the same dataframe. Specifically:

- *n* gives the number of datapoints used in the simulation
- *p* gives the number of covariates used in the simulation

Note that the variable names must be correct for the next parts of this problem. Here is an example of the output this function should return:

> |         |   LASSO |      Ridge |        OLS |   n |   p |
> |:--------|--------:|-----------:|-----------:|----:|----:|
> | Noise 1 |      -0 | -0.0433813 |  0.0236849 | 100 |  50 |
> | Noise 2 |       0 |  0.0284954 |  0.0206112 | 100 |  50 |
> | Noise 3 |       0 |  0.0663444 |  0.069437  | 100 |  50 |
> | Noise 4 |      -0 | -0.106523  | -0.299828  | 100 |  50 |
> | Noise 5 |      -0 | -0.0487962 | -0.0223684 | 100 |  50 |

In [ ]:
def run_simulation(n = 100, p = 50, rho = 0):
  X,y = simulate_data(n=n, p=p, rho = rho)

  # Define the models here. We have done OLS for you;
  # you have to initialize Ridge and LASSO
  ols_regression = linear_model.LinearRegression()

  # Fit the models here. As before, we have done OLS for you.
  ols_regression.fit(X,y)

  df = pd.DataFrame({
    # 'LASSO': ______________,
    # 'Ridge': ______________,
    'OLS': ols_regression.coef_,
    'n' : n,
    'p' : p,
    'rho' : rho,
    },
    index=X.columns)

  return df




### Running the simulations

This code block runs 500 simulations and saves the result into a list. `tqdm.tqdm` is a great way to have a simple progress bar for a long-running task in python (this should take about one minute to run).

We then concatenate all of the dataframes into a long dataframe using `pd.concat`.

In [ ]:
simulations = [run_simulation(70, p = 20) for _ in tqdm(range(100))]

df = pd.concat(simulations)

### Examining Estimates Across Simulations

Use the provided function below to plot the coefficient estimates across simulations. Before you do this, try and guess the answers to the following questions:

- Which of the three methods (OLS, Ridge, Lasso) will do the best at selecting the two predictors that matter most and regularizing the predictors that do not matter?
- Which of the methods will have the highest-variance coefficient estimates?
- How do the coefficients estimated by Ridge, LASSO, and OLS compare to the true values of these parameters marked with the grey lines?

In [ ]:
def make_box_and_whisker_plots(df, population_params=None):
  styles = {
      "LASSO": ("#0072B2", "o"),
      "Ridge": ("#D55E00", "s"),
      "OLS":   ("#009E73", "^"),
  }

  variables = df.index.unique()
  y = np.arange(len(variables))

  fig, ax = plt.subplots(
      figsize=(15, max(3.5, 0.5 * len(variables))),
      layout="constrained",
  )

  for offset, (method, (color, marker)) in zip(
      [-0.2, 0, 0.2], styles.items()
  ):
      samples = [
          df.loc[[variable], method].dropna().to_numpy()
          for variable in variables
      ]

      ax.boxplot(
          samples,
          positions=y + offset,
          orientation="horizontal",
          widths=0.16,
          patch_artist=True,
          manage_ticks=False,
          label=method,
          whis=1.5,
          showfliers=True,
          boxprops=dict(facecolor=color, edgecolor=color, alpha=0.35),
          medianprops=dict(color=color, linewidth=1.6),
          whiskerprops=dict(color=color),
          capprops=dict(color=color),
          flierprops=dict(
              marker=marker, markerfacecolor=color,
              markeredgecolor="none", markersize=3, alpha=0.4,
          ),
      )

  if population_params is not None:
      ax.vlines(
          x=population_params,
          ymin=y - 0.35,
          ymax=y + 0.35,
          colors="grey",
          #linestyles="--",
          linewidths=1.8,
          label="True Value",
          zorder=4,
      )

  ax.axvline(0, color="0.45", linestyle="--", linewidth=1, zorder=0)
  ax.set_yticks(y, labels=variables)
  ax.set_ylim(len(variables) - 0.5, -0.5)
  ax.set_xlabel("Coefficient estimate")
  ax.set_ylabel("Predictor")
  ax.set_title(
      "Regression coefficients across simulations",
      loc="left", fontweight="bold", pad=12,
  )

  ax.set_axisbelow(True)
  ax.grid(axis="x", color="0.9", linewidth=0.8)
  ax.spines[["top", "right", "left"]].set_visible(False)
  ax.tick_params(axis="y", length=0)
  ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(1.02, 1))
  ax.set_xlim(-5,5)

  plt.show()


make_box_and_whisker_plots(df, population_coefficients)

### Reflection

Reflect briefly on the above exercise.

How did the simulation results line up with your expectations? Did you correctly predict which method has the most stable coefficient estimates? How do you think that these results translate into predictive accuracy; that is, which method is likely predict the best for this data-generating process?

Pay special attention to the estimates of the first two coefficients. You should notice that center of the distribution of the OLS coefficients is approximately one for OLS, but shrunk towards zero for ridge and LASSO.  Why is this?



### Highly-Correlated Variables

The function below simulates data by drawing predictors from a multivariate Gaussian distribution. The first two predictors are correlated at level $\rho$, and all other predictors are independent with unit variance.

$$
X_i \sim N(0, \Sigma)
$$

As an example, if we had three predictors, the variance-covariance matrix of the predictors would look like this:


$$
\Sigma =
\begin{pmatrix}
1 & \rho & 0 \\
\rho & 1 & 0 \\
0 & 0 & 1 \\
\end{pmatrix}
$$

If we set $\rho = 0$, then the three predictors are uncorrelated. Because the predictors are multivariate gaussian, this means that they are also independent. As $\rho \to 1$, the variables become increasingly correlated until at $\rho=1$ they are identical.

The outcome variable $y$ is equal to the sum of the first two predictors plus random noise:

$$
Y_i = X_{1i} + X_{2i} + ɛ_i, \quad \varepsilon_i \sim N(0,1)
$$

We simulate one draw from this dataset after we have defined this function with $n=200$, $p=100$, and $\rho=0.995$.

The code chunk below displays the same box-and-whisker plots for this simulation. Notice that the LASSO often completely regularizes the coefficient of predictor 1 or 2 to zero, even though these predictors are both equally important for predicting the outcome. Why is this?  

In answering this question, you should calculate the following quantities:

- The proportion of simulations in which the LASSO shrinks coefficient 1 to zero
- The proportion of simulations in which the LASSO shrinks coefficient 2 to zero
- The proportion of simulations in which the LASSO shrinks both coefficients 1 and 2 to zero

Practitioners will often interpret the predictors that LASSO selects as the 'only predictors that matter.' This is to say, if a predictor is not selected by a LASSO variable selection step, then it must not matter for predicting the outcome. What do you think of this interpretation given the results of the simulation above? Comment and explain your reasoning.

In [ ]:
def simulate_data(n=10_000, p=20, rho = 0.1):
    rng = np.random.default_rng()

    X = rng.standard_normal((n, p))
    X[:, 1] = rho * X[:, 0] + np.sqrt(1 - rho**2) * X[:, 1]
    y = X[:, :2].sum(axis=1) + rng.standard_normal(n)

    X = pd.DataFrame(X, columns=[f"X {i + 1}" for i in range(p)])
    y = pd.Series(y, name="y")
    return X, y


population_coefficients = np.concatenate([
    np.ones(2),
    np.zeros(18),
])


X, y = simulate_data(100, 20)

simulations = [run_simulation(100, p = 100, rho = .995).head(20) for _ in tqdm(range(100))]

df = pd.concat(simulations)

make_box_and_whisker_plots(df, population_coefficients)

In [ ]:
# Space for your answers here

### Problem 2: Risky business (10 points)

In class [(and in these notes)](https://github.com/YData123/sds365-fa22/raw/main/notes/kernel-bias-variance.pdf) we sketched a proof that, when the regression function has two bounded derivatives,
 the bias and variance for kernel smoothing scale as

$$ \text{bias}^2 = O\left(h^4\right)$$
$$ \text{var} = O\left(\frac{1}{nh^p}\right).$$

Here $h$ is the bandwidth parameter, $n$ is the sample size, and $p$ is the number of predictor variables. These expressions are asymptotic, meaning that they apply as $n$ gets large and $h$ gets small.  In this problem your job is to reason about the implications of this bias-variance decomposition for prediction.

*Note:* For this problem, you may either enter your answers in Markdown using $\rm\LaTeX$, or you write them on paper and scan to insert as an image in the notebook; whichever you prefer.


### 2.1 Selecting the optimal bandwidth

Suppose that the bias and variance are such that

$$ \text{bias}^2(\hat m(x))  \leq c_1 h^4 $$
$$ \text{var}(\hat m(x)) \leq c_2 \frac{1}{nh^p}.$$

for two constants $c_1$ and $c_2$. Using these expressions and a little calculus, determine the optimal bandwidth $h$ to minimize the risk function

$$R(h) = {\mathbb E}\left(\hat m(x) - m(x)\right)^2.$$

Your answer should involve the constants $c_1, c_2$, and $n$ and $p$. Give a bound on the resulting risk using this bandwidth.


### 2.2 Bandwith selection without tears

Now, going back to the expressions $\text{bias}^2 = O\left(h^4\right)$ and $ \text{var} = O\left(\displaystyle\frac{1}{nh^p}\right)$, explain why the scaling of the optimal bandwidth (as a function of $n$ and $p$), must satisfy
$\text{bias}^2  \approx \text{var}$; that is, they must be of the same order as $h\to 0$. Then, without using any calculus, use this argument to determine the optimal scaling of the bandwidth and the fastest rate at which the
risk $R(h) = {\mathbb E}\left(\hat m(x) - m(x)\right)^2$ can approach zero as the sample size increases.


### 2.3 The cursed COD

Using the risk bound you derive above, make a plot that demonstrates the curse of dimensionality by plotting the sample size required to achieve a given level of risk. Specifically, let the target risk $R$ vary between 0.1 and 0.5, and let the dimension $p$ vary between 1 and 20, and plot the sample size required to achieve that risk. Give a single plot that shows the collection of curves for each dimension.




In [ ]:
# your code and markdown with derivations here

### Problem 3: A kernel of truth (15 points)

For problem you will implement nonparametric regression using Mercer kernels and penalization, in 1-dimension. This can be compared to regression using smoothing kernels.

As discussed in lecture, nonparametric regression with Mercer kernels is based on the infinite dimensional ridge regression

$$ \hat m = \text{argmin} \| Y - m \|^2 + \lambda \|m\|_K^2$$

By the representer theorem, this is equivalent to setting $\hat m(x) = \sum_{i=1}^n \hat \alpha_i K(X_i, x)$ and
using the finite dimensional optimization

$$ \hat \alpha = \text{argmin} \| Y - {\mathbb K} \alpha \|^2 + \lambda \alpha^T {\mathbb K} \alpha$$

###  3.1 Solve

Derive a closed-form expression for the minimizer $\hat\alpha$. Show all of the steps in your derivation,
and justify each step. (As above, you may either enter your answers in Markdown using $\rm\LaTeX$, or insert an image of your handwritten solution.)


###  3.2 Implement

Next you will use your solution above and implement Mercer kernel regression. We give some starter code.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from IPython.display import clear_output
from time import sleep

The following cell defines some "helper functions" for this exercise. You don't need to change any of this code.
(If you do want to make changes, just describe what you did and why.)

In [ ]:
def plot_estimate(x, f, fhat, X, y, sigma, lmbda, sleeptime=.01):
    clear_output(wait=True)
    plt.figure(figsize=(10,6))
    plt.plot(x, f, color='red', linewidth=2, label='true function')
    plt.plot(x, fhat, color='blue', linewidth=2, label='estimated function')
    plt.scatter(X, y, color='black', alpha=.5, label='random sample')
    plt.ylim(np.min(f)-4*sigma, np.max(f)+4*sigma)
    plt.legend(loc='upper left')
    plt.title('lambda: %.3g' % lmbda)
    plt.xlabel('x')
    plt.ylabel('estimated m(x)')
    plt.show()
    sleep(sleeptime)

def true_fn(x):
    return 3*x**2

def run_simulation(kernel, lmbdas, show_bias_variance=True):
    min_x, max_x = -1, 1
    x = np.linspace(min_x, max_x, 100)
    f = true_fn(x)
    sigma = .25
    estimates = []
    trials = 500

    for lmbda in lmbdas:
        estimates_lambda = []
        for i in np.arange(trials):
            X = np.sort(np.random.uniform(low=min_x, high=max_x, size=50))
            fX = true_fn(X)
            y = fX + sigma*np.random.normal(size=len(X))
            fhat = mercer_kernel_regress(kernel, X, y, x, lmbda=lmbda)
            if i % 50 == 0:
                plot_estimate(x, f, fhat, X, y, sigma, lmbda)
            estimates_lambda.append(fhat)
        estimates.append(estimates_lambda)

    if show_bias_variance == False:
        return

    fhat = np.array(estimates)
    sq_bias = np.zeros(len(lmbdas))
    variance = np.zeros(len(lmbdas))

    for i in np.arange(len(lmbdas)):
        sq_bias[i] = np.mean((np.mean(fhat[i], axis=0) - f)**2)
        variance[i] = np.mean(np.var(fhat[i], axis=0))

    plt.figure(figsize=(10,6))
    plt.plot(lmbdas, sq_bias, label='squared bias', linewidth=2)
    plt.plot(lmbdas, variance, label='variance', linewidth=2)
    plt.plot(lmbdas, sq_bias + variance, label='risk')
    plt.legend()
    plt.show()


Your job is to implement Mercer kernel regression and run it on two
different kernel functions. The two kernels could simply be the Gaussian kernel
with two different bandwidths, or you might experiment with other kernels.

The function `mercer_kernel_regress` takes a kernel, training data `X` and `y`, an array of values `x` to evaluate the function on, and a regularization parameter. You'll use your derivation above to
determine the coefficients $\alpha$. For some clues and suggestions on how to do the
implementation, see our demo code for smoothing kernels. You need to do something very similar.


In [ ]:
def mercer_kernel_regress(kernel, X, y, x, lmbda):
    # your implementation here
    _

def kernel1(x,y):
    # your implementation here
    _


def kernel2(x,y):
    # your implementation here
    _

###  3.3 Run two simulations and select regularization parameters

Finally, using our starter code and your own implementation above, run two simulations, one
using `kernel1` and the other using `kernel2`. After each simulation, select a regularization level from the bias-variance tradeoff, and then run a final simulation with that regularization level. In the following
starter code, you only need to specify the sequence of regularization parameters.


In [ ]:
lmbdas = # define your sequence of lambdas
run_simulation(kernel1, lmbdas)

In [ ]:
lambda_hat = # set the optimal lambda
run_simulation(kernel1, [lambda_hat], show_bias_variance=False)

In [ ]:
lmbdas = # define your sequence of lambdas
run_simulation(kernel2, lmbdas)

### Problem 4: An algebraic simplification of LOOCV (15 points)

Leave-One-Out Cross Validation (LOOCV) is a specific type of
$K$-fold cross validation where $K$ equals the number of observations in the dataset.
It works as follows for a training set with $n$ observations:

1. A single observation is used as the validation set,
    and the remaining $n-1$ observations serve as the training set.
2. A model is trained on the $n-1$ observations and
    validated on the single left-out observation.
3. This process is repeated $n$ times, each time leaving out a different
    observation as the validation set.
4. The LOOCV error is then the average error across all $n$ trials.

LOOCV is particularly useful because:
- It utilizes almost all the data for training,
    so it's less prone to high variance compared to other validation schemes.
- Since each observation is tested exactly once,
    LOOCV provides a very thorough out-of-sample testing mechanism.

However, it can be computationally expensive because you have to fit the model $n$ times.
    Luckily, for some models, there are algebraic simplifications available
    that make it computationally efficient.
    Expressing LOOCV in terms of the hat matrix allows for efficient
    computation of the LOOCV error without the need to refit the model for
    each left-out observation, making it a valuable tool for model evaluation.

Recall that the LOOCV error can be expressed as:

$$ LOOCV = \frac{1}{n} \sum_{i=1}^{n} \left( y_i - \hat{y}_{-i} \right)^2, $$

where $\hat{y}_{-i}$ represents the prediction for the $i^{th}$ observation
when it's left out from the training process.
In the following questions, you will be deriving an alternative expression
of the LOOCV error for both kernel and ridge regression, following the hints below.

### 1. LOOCV for kernel smoothing:

For kernels, we know that the LOOCV error can be equivalently written as the following form:

$$ LOOCV = \frac{1}{n} \sum_{i=1}^{n} \left( \frac{y_i - \hat y_i}{1 - L_{ii}} \right)^2, $$
where $\hat y_i$ is the predicted value from the model fit on all data, and
$L_{ii}$ is the $i^{th}$ diagonal element of the hat matrix $L$.

For kernel regression, we have
$$ \hat{y} = L y, $$
where
- $ \hat{y} $ is the vector of predictions.
- $ y $ is the observed response values.
- $ L $ is the hat matrix and is defined by the kernel (for a given bandwidth).
So, each diagonal element $ L_{ii} $ of the matrix $ L $ is defined as:
$$ L_{ii} = \frac{K\left(x_i, x_i\right)}{\sum_{j=1}^{n} K\left(x_i, x_j\right)}, $$
where
- $ K $ is the kernel function.
- $ x_i $ and $ x_j $ are the predictor values for observations $ i $ and $ j $, respectively.

The diagonal elements $ L_{ii} $ give the "leverage" of each observation, which can be interpreted as the influence an observation has on its own prediction.

Derive this alternative expression of the LOOCV error for kernel regression. That's to say, for kernel regression, prove that

$$ y_i - \hat{y}_{-i}  =  \frac{y_i - \hat y_i}{1 - L_{ii}}$$

In [ ]:
# Your markdown here.

### Problem 5: LASSO and Elastic Net (10 points)

In class, we derived the solution for the one-dimensional LASSO using convexity and subgradients. We also know that coordinate descent can be used to solve LASSO in the high-dimensional case. Sometimes, we introduce a penalty that is a mixture of the $\ell_1$ and $\ell_2$ norms,  

$$
\lambda_1 \|\beta\|_1 + \lambda_2 \|\beta\|_2^2,
$$  

which is referred to as the **Elastic Net**. This problem demonstrates how the Elastic Net can be computed using the LASSO.

Let us first recall the one-dimensional LASSO objective:

$$
f(\beta) = \frac{a}{2}\beta^2 - b\beta + \lambda |\beta|,
\qquad a > 0,\; \lambda \ge 0.
$$

The minimizer is given by

$$
\widehat{\beta} = \frac{\operatorname{sign}(b)\,\big(|b| - \lambda\big)_+}{a},
$$

where $(x)_+ = \max\{x,0\}$.

5.1 Solve the one-dimensional LASSO problem

$$
\widehat{\beta} = \arg\min_\beta \left\{ \frac{1}{2n}\sum_{i=1}^n (y_i - \beta x_i)^2 + \lambda |\beta| \right\},
$$

where $\lambda \geq 0$ and at least one $x_i \neq 0$.

In [ ]:
# Your markdown here.

5.2 Solve the one-dimensional Elastic Net
$$
\widehat{\beta} = \arg\min_\beta \left\{ \frac{1}{2n} \sum_{i=1}^n (y_i - \beta x_i)^2 + \lambda_1 |\beta| + \lambda_2 \beta^2 \right\},
$$

where $\lambda_1 \geq 0$, $\lambda_2 \geq 0$, and at least one $x_i \neq 0$.  

*Hint:* Convert this problem into the form of 5.1. What happens if we observe a new data point $(\sqrt{2n\lambda_2}, 0)$?

In [ ]:
# Your markdown here.

5.3 Now consider the case where we have data $\mathbf{Y} \in \mathbb{R}^n$ and $\mathbf{X} \in \mathbb{R}^{n \times p}$. We want to find an estimator

$$
\widehat{\beta} = \arg\min_\beta \left\{ \frac{1}{2n}\| \mathbf{Y} - \mathbf{X}\beta\|_2^2 + \lambda_1 \|\beta\|_1 + \lambda_2 \|\beta\|_2^2 \right\}.
$$

Assume we have access to an oracle that can solve the LASSO. How can we use it to solve the Elastic Net?  

*Hint:* Construct augmented variables $\mathbf{Y}^*$ and $\mathbf{X}^*$
(not necessarily the same shapes as $\mathbf{Y}$ and $\mathbf{X}$) and show that the objective is equivalent to
$$
\frac{1}{2\,\mathrm{len}(\mathbf{Y}^*)}
\|\mathbf{Y}^* - \mathbf{X}^*\beta\|_2^2
+ \lambda_1 \|\beta\|_1.
$$

In [ ]:
# Your markdown here.